# AI 102 - GenAI Creator Lab: Solution Notebook

Run top-to-bottom in Google Colab. This builds, trains and runs a small GPT-style language model from scratch, then demonstrates sampling. See the lab content for explanations.

## Phase 0 - Environment

In [ ]:
#@title Setup dependencies
!pip install -q datasets

import os, re, time
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from datasets import load_dataset

In [ ]:
#@title Check GPU
device_name = tf.test.gpu_device_name()
if device_name:
    print("GPU available:", device_name)
else:
    print("No GPU detected. Training will still work, but slower.")

## Phase 1 - Tokenization & Embeddings

In [ ]:
#@title Load the tiny_shakespeare corpus
dataset = load_dataset("tiny_shakespeare", trust_remote_code=True)
text = dataset["train"][0]["text"]
print("Corpus length (characters):", len(text))
print(text[:300])

In [ ]:
#@title Inspect the character vocabulary
chars = sorted(list(set(text)))
vocab_size = len(chars)
print("Vocab size:", vocab_size)
print("Characters:", "".join(chars))

In [ ]:
#@title Build char <-> int mappings
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for i, ch in enumerate(chars)}
def encode(s): return [stoi[c] for c in s]
def decode(ids): return "".join(itos[i] for i in ids)

sample = "To be, or not to be"
print(sample, "->", encode(sample), "->", decode(encode(sample)))

In [ ]:
#@title Encode the full corpus
data = tf.constant(encode(text), dtype=tf.int32)
print("Encoded corpus shape:", data.shape)

## Phase 2 - The Decoder Block

In [ ]:
#@title Define a single decoder (Transformer) block
class DecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout_rate=0.1):
        super().__init__()
        self.attn = layers.MultiHeadAttention(num_heads=num_heads, key_dim=embed_dim)
        self.ffn = keras.Sequential([
            layers.Dense(ff_dim, activation="gelu"),
            layers.Dense(embed_dim),
        ])
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout_rate)
        self.drop2 = layers.Dropout(dropout_rate)

    def call(self, x, training=False):
        seq_len = tf.shape(x)[1]
        mask = tf.linalg.band_part(tf.ones((seq_len, seq_len), dtype=tf.bool), -1, 0)
        a = self.ln1(x)
        attn_out = self.attn(query=a, value=a, key=a, attention_mask=mask)
        x = x + self.drop1(attn_out, training=training)
        ffn_out = self.ffn(self.ln2(x))
        x = x + self.drop2(ffn_out, training=training)
        return x

## Phase 3 - Build, Compile & Train

In [ ]:
#@title Hyperparameters (tuning knobs)
embed_dim   = 64
num_heads   = 4
ff_dim      = 256
num_blocks  = 4
dropout_rate = 0.1
block_size  = 128
batch_size  = 64
learning_rate = 3e-4

In [ ]:
#@title Build the GPT-style model
def build_gpt():
    inputs = keras.Input(shape=(None,), dtype="int32")
    seq_len = tf.shape(inputs)[1]
    tok = layers.Embedding(vocab_size, embed_dim)(inputs)
    pos = layers.Embedding(block_size, embed_dim)(tf.range(seq_len))
    x = tok + pos
    for _ in range(num_blocks):
        x = DecoderBlock(embed_dim, num_heads, ff_dim, dropout_rate)(x)
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    logits = layers.Dense(vocab_size)(x)
    return keras.Model(inputs=inputs, outputs=logits)

model = build_gpt()
model.summary()

In [ ]:
#@title Compile
loss_fn = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
model.compile(optimizer=optimizer, loss=loss_fn)

In [ ]:
#@title Batching: (input, target) shifted by one
def get_batch(data, batch_size, block_size):
    max_start = len(data) - block_size - 1
    starts = tf.random.uniform((batch_size,), 0, max_start, dtype=tf.int32)
    x = tf.stack([data[i : i + block_size] for i in starts])
    y = tf.stack([data[i + 1 : i + block_size + 1] for i in starts])
    return x, y

xb, yb = get_batch(data, batch_size, block_size)
print("x:", xb.shape, "y:", yb.shape)
print("input :", repr(decode(xb[0][:40].numpy())))
print("target:", repr(decode(yb[0][:40].numpy())))

In [ ]:
#@title Train
steps = 3000
eval_every = 200

@tf.function
def train_step(x, y):
    with tf.GradientTape() as tape:
        logits = model(x, training=True)
        loss = loss_fn(y, logits)
    grads = tape.gradient(loss, model.trainable_variables)
    optimizer.apply_gradients(zip(grads, model.trainable_variables))
    return loss

history = []
start = time.time()
for step in range(1, steps + 1):
    xb, yb = get_batch(data, batch_size, block_size)
    loss = train_step(xb, yb)
    if step % eval_every == 0 or step == 1:
        history.append((step, float(loss)))
        print(f"step {step:5d} | loss {float(loss):.4f} | {time.time()-start:6.1f}s")
print("Training complete.")

In [ ]:
#@title Plot loss
sx = [h[0] for h in history]; sy = [h[1] for h in history]
plt.plot(sx, sy, marker="o"); plt.title("Training loss")
plt.xlabel("Step"); plt.ylabel("Cross-entropy"); plt.grid(True); plt.show()

In [ ]:
#@title Save foundation model
model.save("genai_foundation.keras")
print("Saved genai_foundation.keras")

## Phase 4 - Inference & Sampling

In [ ]:
#@title Generate text
def generate(model, prompt, max_new_tokens=300, temperature=1.0, top_k=None):
    ids = encode(prompt)
    for _ in range(max_new_tokens):
        context = tf.constant([ids[-block_size:]], dtype=tf.int32)
        logits = model(context, training=False)
        logits = logits[0, -1, :] / temperature
        if top_k is not None:
            values, _ = tf.math.top_k(logits, k=top_k)
            min_keep = values[-1]
            logits = tf.where(logits < min_keep, tf.fill(tf.shape(logits), -1e9), logits)
        probs = tf.nn.softmax(logits).numpy()
        next_id = np.random.choice(len(probs), p=probs)
        ids.append(int(next_id))
    return decode(ids)

print(generate(model, "ROMEO:", max_new_tokens=400, temperature=0.8, top_k=40))

In [ ]:
#@title Compare sampling settings
for t in [0.2, 0.8, 1.4]:
    print(f"=== temperature={t} ===")
    print(generate(model, "ROMEO:", max_new_tokens=160, temperature=t))
    print()

## Phase 6 - Challenge: validation split & scoring

In [ ]:
#@title Validation split + loss estimate
n = int(0.9 * len(data))
train_data = data[:n]; val_data = data[n:]
print("Train tokens:", len(train_data), "| Val tokens:", len(val_data))

def estimate_val_loss(model, val_data, iters=50):
    losses = []
    for _ in range(iters):
        xb, yb = get_batch(val_data, batch_size, block_size)
        logits = model(xb, training=False)
        losses.append(float(loss_fn(yb, logits)))
    return float(np.mean(losses))

vl = estimate_val_loss(model, val_data)
print("Validation loss:", round(vl, 4), "| Perplexity:", round(float(np.exp(vl)), 2))